# Stage 1 — Data Ingestion

**Project:** Cleveland Heart Disease — Tier A diagnostic + predictive analysis
**Notebook 1 of 6** · Governed by `DOCS/STRUCTURE.md` Stage 1

---

## What this notebook is for

Acquire `heart.csv` into the analysis environment, prove what it actually contains, and record
an immutable copy plus its lineage. **No cleaning happens here** — Stage 2 owns every
modification. The job of this notebook is to make the file's defects *visible and dated*, so
nothing downstream can quietly assume the data is clean.

### What we expect to find (from `DOCS/heart_dataset_analysis.md`)

1. 1,025 rows × 14 columns, zero nulls.
2. **Massive duplication** — only ~302 unique patients.
3. **Out-of-dictionary category codes** — `thal = 0` and `ca = 4`, which the data dictionary
   does not permit. These are UCI `?` missing-value placeholders coerced into numbers.

Items 2 and 3 are the reason this notebook exists in its current form. A schema check that
passes everything would be theatre; this one is built to fail where the data is broken.

In [1]:
import hashlib
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42

# Resolve the project root whether this runs from the project folder or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "heart.csv").exists() else Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

print(f"python  {platform.python_version()}")
print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")
print(f"root    {ROOT.name}/")

python  3.14.6
pandas  3.0.3
numpy   2.5.1
root    Heart Disease/


## 1.1 Load the source file

Single flat CSV, comma-delimited, header row present.

In [2]:
SOURCE = ROOT / "heart.csv"
df = pd.read_csv(SOURCE)

print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

shape: 1,025 rows x 14 columns


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [3]:
# Structural summary: dtype, non-null count, distinct values, range
structure = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "nulls": df.isna().sum(),
    "n_distinct": df.nunique(),
    "min": df.min(numeric_only=True),
    "max": df.max(numeric_only=True),
})
structure

,dtype,non_null,nulls,n_distinct,min,max
age,int64,1025,0,41,29.0,77.0
sex,int64,1025,0,2,0.0,1.0
cp,int64,1025,0,4,0.0,3.0
trestbps,int64,1025,0,49,94.0,200.0
chol,int64,1025,0,152,126.0,564.0
fbs,int64,1025,0,2,0.0,1.0
restecg,int64,1025,0,3,0.0,2.0
thalach,int64,1025,0,91,71.0,202.0
exang,int64,1025,0,2,0.0,1.0
oldpeak,float64,1025,0,40,0.0,6.2


**Observation.** Every column is numeric and every column is complete — `nulls = 0` throughout.

> **This is the file's most misleading property.** Zero nulls does not mean zero missingness. The
> original UCI records used `?` for unknown values; somewhere upstream those were coerced into
> numeric codes. Section 1.3 tests for exactly that.

## 1.2 Lineage record

Source path, SHA-256 content hash, pull timestamp, and shape — so any future run can prove it
read the same bytes.

In [4]:
file_bytes = SOURCE.read_bytes()
lineage = {
    "source_path": SOURCE.name,
    "source_description": "UCI Cleveland Heart Disease (14-attribute subset), as delivered",
    "sha256": hashlib.sha256(file_bytes).hexdigest(),
    "size_bytes": len(file_bytes),
    "pull_timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "n_rows_raw": df.shape[0],
    "n_cols_raw": df.shape[1],
    "columns": "|".join(df.columns),
}
pd.DataFrame([lineage]).T.rename(columns={0: "value"})

,value
source_path,heart.csv
source_description,UCI Cleveland Heart Disease (14-attribute subs...
sha256,ddb2996b2f4db2e00aad13f4518200179ff69f79093838...
size_bytes,38114
pull_timestamp_utc,2026-07-26T15:41:32+00:00
n_rows_raw,1025
n_cols_raw,14
columns,age|sex|cp|trestbps|chol|fbs|restecg|thalach|e...


In [5]:
pd.DataFrame([lineage]).to_csv(RAW_DIR / "_lineage.csv", index=False)
print(f"wrote {RAW_DIR.relative_to(ROOT)}/_lineage.csv")

wrote data\raw/_lineage.csv


## 1.3 Schema validation against the data dictionary

Two rule families, evaluated per column:

- **Domain rules** — categorical and binary columns must contain *only* the codes the data
  dictionary permits. A value outside the permitted set is either an encoding error or hidden
  missing data; either way it is a defect.
- **Plausibility rules** — continuous measurements must fall inside clinically possible ranges.

Every rule is recorded with its verdict, including the ones that fail. Failures are *carried
forward as open items* for Stage 2, not silently repaired here.

In [6]:
# (column, kind, permitted set or (lo, hi), source of the rule)
DOMAIN_RULES = {
    "sex":     ({0, 1},          "0 = female, 1 = male"),
    "cp":      ({0, 1, 2, 3},    "chest pain type, 4 nominal categories"),
    "fbs":     ({0, 1},          "fasting blood sugar > 120 mg/dl"),
    "restecg": ({0, 1, 2},       "resting ECG result"),
    "exang":   ({0, 1},          "exercise-induced angina"),
    "slope":   ({0, 1, 2},       "slope of peak exercise ST segment"),
    "ca":      ({0, 1, 2, 3},    "major vessels coloured by fluoroscopy (0-3)"),
    "thal":    ({1, 2, 3},       "thallium scan: 1 normal, 2 fixed, 3 reversible defect"),
    "target":  ({0, 1},          "0 = no disease, 1 = disease"),
}

RANGE_RULES = {
    "age":      ((18, 100),  "adult patient age, years"),
    "trestbps": ((80, 220),  "resting systolic BP, mm Hg"),
    "chol":     ((100, 600), "serum cholesterol, mg/dl"),
    "thalach":  ((60, 220),  "max heart rate achieved, bpm"),
    "oldpeak":  ((0, 7),     "ST depression vs rest, mm"),
}

rows = []
for col, (allowed, note) in DOMAIN_RULES.items():
    observed = set(df[col].unique())
    illegal = sorted(observed - allowed)
    counts = {int(v): int((df[col] == v).sum()) for v in illegal}
    rows.append({
        "column": col, "rule": "domain",
        "expected": "{" + ",".join(str(v) for v in sorted(allowed)) + "}",
        "observed": "{" + ",".join(str(v) for v in sorted(observed)) + "}",
        "violations": "; ".join(f"{k} x{v}" for k, v in counts.items()) or "-",
        "n_violating_rows": int(sum(counts.values())),
        "verdict": "PASS" if not illegal else "FAIL",
        "note": note,
    })

for col, ((lo, hi), note) in RANGE_RULES.items():
    bad = int(((df[col] < lo) | (df[col] > hi)).sum())
    rows.append({
        "column": col, "rule": "plausible_range",
        "expected": f"[{lo}, {hi}]",
        "observed": f"[{df[col].min():g}, {df[col].max():g}]",
        "violations": "-" if bad == 0 else f"{bad} rows outside range",
        "n_violating_rows": bad,
        "verdict": "PASS" if bad == 0 else "FAIL",
        "note": note,
    })

schema = pd.DataFrame(rows)
schema.to_csv(RAW_DIR / "_schema_validation.csv", index=False)
schema

,column,rule,expected,observed,violations,n_violating_rows,verdict,note
0,sex,domain,"{0,1}","{0,1}",-,0,PASS,"0 = female, 1 = male"
1,cp,domain,"{0,1,2,3}","{0,1,2,3}",-,0,PASS,"chest pain type, 4 nominal categories"
2,fbs,domain,"{0,1}","{0,1}",-,0,PASS,fasting blood sugar > 120 mg/dl
3,restecg,domain,"{0,1,2}","{0,1,2}",-,0,PASS,resting ECG result
4,exang,domain,"{0,1}","{0,1}",-,0,PASS,exercise-induced angina
5,slope,domain,"{0,1,2}","{0,1,2}",-,0,PASS,slope of peak exercise ST segment
6,ca,domain,"{0,1,2,3}","{0,1,2,3,4}",4 x18,18,FAIL,major vessels coloured by fluoroscopy (0-3)
7,thal,domain,"{1,2,3}","{0,1,2,3}",0 x7,7,FAIL,"thallium scan: 1 normal, 2 fixed, 3 reversible..."
8,target,domain,"{0,1}","{0,1}",-,0,PASS,"0 = no disease, 1 = disease"
9,age,plausible_range,"[18, 100]","[29, 77]",-,0,PASS,"adult patient age, years"


In [7]:
n_fail = int((schema["verdict"] == "FAIL").sum())
print(f"schema rules evaluated : {len(schema)}")
print(f"passed                 : {len(schema) - n_fail}")
print(f"FAILED                 : {n_fail}\n")
if n_fail:
    for _, r in schema[schema["verdict"] == "FAIL"].iterrows():
        print(f"  [FAIL] {r['column']:<9} expected {r['expected']:<12} "
              f"got {r['observed']:<12} -> {r['violations']}")

schema rules evaluated : 14
passed                 : 12
FAILED                 : 2

  [FAIL] ca        expected {0,1,2,3}    got {0,1,2,3,4}  -> 4 x18
  [FAIL] thal      expected {1,2,3}      got {0,1,2,3}    -> 0 x7


### Finding — two columns carry hidden missing data

`thal` and `ca` both contain codes the data dictionary does not define:

| Column | Permitted | Illegal code | Rows affected (raw file) |
|---|---|---|---|
| `thal` | {1, 2, 3} | `0` | 7 |
| `ca` | {0, 1, 2, 3} | `4` | 18 |

**So what:** these are almost certainly the original UCI `?` placeholders, coerced to `0` and `4`
during whatever pipeline produced this file. They are *missing values disguised as valid
categories*. Left alone, they would be modelled as a real thallium result and a real fifth vessel
count — inventing clinical facts that were never measured.

**Implication:** Stage 2 recodes both to `NaN`. The counts above are inflated by the file's
duplication (Section 1.4); the true patient-level counts are established after deduplication.

All continuous measurements pass their plausibility ranges — the extremes (`chol` 564,
`trestbps` 200) are high but clinically real, and are retained.

## 1.4 Duplication — raised here, resolved in Stage 2

The single most consequential property of this file. Recording it at first contact so the
finding is dated to ingestion rather than discovered late.

In [8]:
n_total = len(df)
n_unique = len(df.drop_duplicates())
n_dupe_rows = int(df.duplicated().sum())

print(f"rows in file            : {n_total:,}")
print(f"distinct rows           : {n_unique:,}")
print(f"exact duplicate rows    : {n_dupe_rows:,}  ({n_dupe_rows / n_total:.1%} of the file)")
print(f"mean copies per patient : {n_total / n_unique:.2f}x")

rows in file            : 1,025
distinct rows           : 302
exact duplicate rows    : 723  (70.5% of the file)
mean copies per patient : 3.39x


In [9]:
# Does the duplication distort the label balance? (Raw vs distinct rows)
label = pd.DataFrame({
    "raw_file": df["target"].value_counts().sort_index(),
    "distinct_rows": df.drop_duplicates()["target"].value_counts().sort_index(),
})
label.loc["total"] = label.sum()
label["raw_share"] = (label["raw_file"] / label.loc["total", "raw_file"]).map("{:.1%}".format)
label["distinct_share"] = (label["distinct_rows"] / label.loc["total", "distinct_rows"]).map("{:.1%}".format)
label.index = ["0 = no disease", "1 = disease", "total"]
label

,raw_file,distinct_rows,raw_share,distinct_share
0 = no disease,499,138,48.7%,45.7%
1 = disease,526,164,51.3%,54.3%
total,1025,302,100.0%,100.0%


### Finding — the file is 302 patients wearing 1,025 name tags

723 of 1,025 rows (70.5%) are exact copies. The effective sample size is **302**, and the
published Cleveland cohort is 303 patients — so this file is the standard Cleveland set,
replicated ~3.4× on average.

**So what:** any train/test split performed on the 1,025-row file will place identical patient
records on both sides of the split. The model does not generalise; it recognises rows it has
already memorised. Every accuracy figure published against this file in that configuration is
measuring contamination.

**Implication:** all analysis in this project runs on the **302 distinct patients**. The full
file is retained for exactly one purpose — as the contaminated comparator in the Stage 5b
leakage experiment, which quantifies the inflation in AUC points.

Note also that the label balance *shifts* under deduplication (51.3% → 54.3% positive). The
copying was therefore **not uniform** across rows — Stage 2 characterises this.

## 1.5 Data governance

The file is public, de-identified UCI research data. But two columns are protected attributes,
and that determines an obligation later in the pipeline.

In [10]:
governance = pd.DataFrame([
    {"item": "PII / direct identifiers", "status": "NONE",
     "detail": "No name, patient ID, MRN, address, or date. De-identified by design."},
    {"item": "Timestamps / linkage keys", "status": "NONE",
     "detail": "No dates - no temporal drift check and no re-identification vector."},
    {"item": "Protected attribute: sex", "status": "PRESENT",
     "detail": "Binary sex is a model input. Triggers a mandatory subgroup fairness audit."},
    {"item": "Protected attribute: age", "status": "PRESENT",
     "detail": "Age is a model input. Audited by band alongside sex."},
    {"item": "Special-category data (health)", "status": "PRESENT",
     "detail": "Clinical measurements + diagnosis. Public research dataset; consent handled by "
               "the original 1988 collection. Analysis use only."},
    {"item": "Purpose limitation", "status": "OK",
     "detail": "Methodological benchmarking and association analysis. No deployment, no "
               "individual patient decision-making."},
])
governance

,item,status,detail
0,PII / direct identifiers,NONE,"No name, patient ID, MRN, address, or date. De..."
1,Timestamps / linkage keys,NONE,No dates - no temporal drift check and no re-i...
2,Protected attribute: sex,PRESENT,Binary sex is a model input. Triggers a mandat...
3,Protected attribute: age,PRESENT,Age is a model input. Audited by band alongsid...
4,Special-category data (health),PRESENT,Clinical measurements + diagnosis. Public rese...
5,Purpose limitation,OK,Methodological benchmarking and association an...


**So what:** unlike most anonymised clinical benchmarks, this one *does* carry protected
attributes. Under `DOCS/STRUCTURE.md` Cross-Cutting rules, a model that affects people and has
access to a protected attribute owes a **subgroup fairness audit**. That audit is therefore
mandatory in Stage 6 and cannot be waived — it is scheduled in `05_modeling.ipynb`.

In [11]:
governance.to_csv(RAW_DIR / "_governance.csv", index=False)

# Immutable copy - never edited after this cell
df.to_csv(RAW_DIR / "heart.csv", index=False)
print(f"wrote {RAW_DIR.relative_to(ROOT)}/_governance.csv")
print(f"wrote {RAW_DIR.relative_to(ROOT)}/heart.csv   <- immutable raw copy, never edited")

wrote data\raw/_governance.csv
wrote data\raw/heart.csv   <- immutable raw copy, never edited


---

## Stage 1 — Gate Checklist

- [x] **Raw data saved to `data/raw/`** (untouched copy — `data/raw/heart.csv`)
- [x] **Source, timestamp, and row/column counts logged** (`_lineage.csv`, incl. SHA-256)
- [x] **Schema matches expectations, or discrepancies are documented** — 14 rules evaluated,
      **2 documented FAILures** (`thal = 0`, `ca = 4`) carried forward to Stage 2 as open items
- [x] **PII / data governance review completed** (`_governance.csv`) — de-identified, but
      `sex` and `age` are protected attributes, which makes a Stage 6 fairness audit mandatory

### Open items handed to Stage 2

| # | Item | Action required |
|---|---|---|
| 1 | 723 duplicate rows (70.5%) | Deduplicate to 302 patients; characterise the copying pattern |
| 2 | `thal = 0` — outside {1,2,3} | Recode to `NaN`; decide handling; count at patient level |
| 3 | `ca = 4` — outside {0,1,2,3} | Recode to `NaN`; decide handling; count at patient level |
| 4 | Label balance shifts under dedup | Confirm non-uniform copying; use the 302-row balance |

**Next:** `02_cleaning.ipynb` — deduplication forensics and the placeholder recoding.